Recommenders: DLRM, DCN-v2, SASRec, and TIGER Semantic IDs

Deep learning architectures and generative paradigms for industrial recommendation:
1. DLRM Feature Interaction: Explicit pairwise dot products between dense and sparse embeddings.
2. DCN-v2 Cross Layers: Polynomial-degree explicit feature crossing with linear complexity.
3. SASRec: Causally-masked self-attention over historical user interaction sequences.
4. TIGER Semantic IDs: Hierarchical residual vector quantization for generative recommendation.

## 2. Mathematical Formulations & Theoretical Foundations

- **DLRM Feature Interaction**:
  $$M = V V^T, \quad \text{Interactions} = \{ M_{i, j} \}_{i > j}, \quad \text{Out} = [v_0, \text{Interactions}]$$
- **DCN-v2 Explicit Cross Layer**:
  $$x_{l+1} = x_0 \odot (x_l W_l + b_l) + x_l$$
- **SASRec Causal Self-Attention**:
  $$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{Q K^T}{\sqrt{D}} + M_{\text{causal}}\right) V$$
- **RQ-VAE Quantization Step**:
  $$c_k = \arg\min_v \|r_{k-1} - \mathcal{C}_{k, v}\|^2, \quad r_k = r_{k-1} - \mathcal{C}_{k, c_k}$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "07_domain_tasks" / "11_recommenders_dlrm_dcnv2_sasrec_and_tiger_semantic_ids.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. DLRM Feature Interaction
def dlrm_feature_interaction(v_dense: torch.Tensor, v_sparse: torch.Tensor) -> torch.Tensor:
    s = v_sparse.shape[1]
    v_all = torch.cat([v_dense.unsqueeze(1), v_sparse], dim=1)
    interaction_mat = torch.bmm(v_all, v_all.transpose(1, 2))
    tril_indices = torch.tril_indices(s + 1, s + 1, offset=-1)
    interactions = interaction_mat[:, tril_indices[0], tril_indices[1]]
    return torch.cat([v_dense, interactions], dim=1)

dlrm_calc = dlrm_feature_interaction(golden["v_dense"], golden["v_sparse"])
diff_dlrm = torch.max(torch.abs(dlrm_calc - golden["dlrm_ref"])).item()
print(f"DLRM Feature Interaction max abs error: {diff_dlrm:.2e}")
assert diff_dlrm < 1e-5

In [ ]:
# 2. DCN-v2 Cross Layer
def dcn_v2_cross_layer(x0: torch.Tensor, xl: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    proj = xl @ weight + bias
    return x0 * proj + xl

dcn_calc = dcn_v2_cross_layer(golden["x0"], golden["xl"], golden["w_cross"], golden["b_cross"])
diff_dcn = torch.max(torch.abs(dcn_calc - golden["dcn_ref"])).item()
print(f"DCN-v2 Cross Layer max abs error: {diff_dcn:.2e}")
assert diff_dcn < 1e-5

In [ ]:
# 3. SASRec Causal Self-Attention
def sasrec_causal_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    d = q.shape[-1]
    t = q.shape[1]
    scores = torch.bmm(q, k.transpose(1, 2)) / (d ** 0.5)
    mask = torch.triu(torch.ones(t, t, device=q.device), diagonal=1).bool()
    scores = scores.masked_fill(mask.unsqueeze(0), -1e9)
    attn_weights = F.softmax(scores, dim=-1)
    return torch.bmm(attn_weights, v)

sasrec_calc = sasrec_causal_attention(golden["q"], golden["k"], golden["v"])
diff_sasrec = torch.max(torch.abs(sasrec_calc - golden["sasrec_ref"])).item()
print(f"SASRec Causal Attention max abs error: {diff_sasrec:.2e}")
assert diff_sasrec < 1e-5

In [ ]:
# 4. RQ-VAE Quantize Step & Summary Table
def rq_vae_quantize_step(residual: torch.Tensor, codebook: torch.Tensor):
    r_sq = torch.sum(residual ** 2, dim=-1, keepdim=True)
    c_sq = torch.sum(codebook ** 2, dim=-1, keepdim=True).t()
    dist = r_sq - 2.0 * (residual @ codebook.t()) + c_sq
    indices = torch.argmin(dist, dim=-1)
    quantized = codebook[indices]
    next_residual = residual - quantized
    return quantized, indices, next_residual

quant_calc, idx_calc, next_res_calc = rq_vae_quantize_step(golden["residual"], golden["codebook"])
diff_quant = torch.max(torch.abs(quant_calc - golden["quantized_ref"])).item()
diff_idx = int(torch.max(torch.abs(idx_calc - golden["indices_ref"])).item())
diff_res = torch.max(torch.abs(next_res_calc - golden["next_res_ref"])).item()

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'DLRM Feature Interaction':<32} | {diff_dlrm:<15.2e} | PASSED")
print(f"{'DCN-v2 Cross Layer':<32} | {diff_dcn:<15.2e} | PASSED")
print(f"{'SASRec Causal Attention':<32} | {diff_sasrec:<15.2e} | PASSED")
print(f"{'RQ-VAE Quantization':<32} | {diff_quant:<15.2e} | PASSED")
print("=" * 65)
assert diff_quant < 1e-5
assert diff_idx == 0
assert diff_res < 1e-5

## 8. Summary, Invariants & Diagnostic Checklist

- [x] DLRM explicitly computes pairwise inner products between all dense and sparse embeddings.
- [x] DCN-v2 captures all bounded-degree polynomial crosses with linear complexity in input dimension.
- [x] SASRec leverages causal attention to adaptively weigh past user behaviors for next-item prediction.
- [x] TIGER transforms continuous catalog representations into hierarchical semantic discrete codes via RQ-VAE.